# Step 11 — Content-profile similarity in the final model

**Why this step.** Step 9 (Jieming) built 13 content-profile similarity columns (`cps_*`: cosine between a candidate's name / brand / category terms and the products in the user's history) and showed, with a fixed 300-tree LightGBM on validation slates, a gain of +0.0027 NDCG@10 (95% CI 0.0015–0.0038), concentrated on all-cold slates. That protocol cannot say whether the **final model** improves: the final model is the Step 6 ensemble of a tuned LightGBM (learning rate 0.03, up to 1,200 trees, early stopping) and a 3-seed neural ranker that already reads the user's history through LSA embeddings and attention, and it is scored on the test set.

**What this notebook does.**

1. Adds the Step 9 columns to the Step 6 LightGBM in three variants (all 13 columns; all 13 replacing the three LSA profile columns; the five strongest replacing the LSA columns) and **chooses on validation**.
2. Scores the chosen variant **once on test**, with the paired bootstrap of Step 6, overall and by cold / mixed / warm slate.
3. Rebuilds the final ensemble with the Step 6 neural test scores unchanged and only the LightGBM half replaced. Step 6 chose the blend (within-slate z-scores, weight 0.5) on validation; that choice is kept, so nothing is selected on test. If the new ensemble beats the published 0.4201, the features belong in the final model.

**What it cannot do.** The neural half is not retrained here (no GPU, and Step 6 saved only its test scores). If the LightGBM half gains, the full route is Step 6 with `JD_CPS=1`, which retrains both models with the Step 9 columns and re-chooses the blend weight on validation (Section 7).

**Contract.** Run after Steps 6 and 9. Reads `features.parquet`, `processed/feature_content_profile/content_profile_features.parquet` (Step 9) and `step6_test_predictions.parquet`; modifies none of them. Writes `step11_results.csv`, `step11_test_predictions.parquet` and `step11_chosen_features.json`. Colab: High-RAM CPU runtime, about 40 minutes. Set `JD_SMOKE=1` for a 2% local run (needs Step 9 run in smoke mode first).

In [1]:
import os, json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.dataset as ds
import warnings
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'
SFX = '_smoke' if SMOKE else ''
SEED = 42
CPS_PATH = OUT_DIR / 'feature_content_profile' / f'content_profile_features{SFX}.parquet'     # Step 9
PRED_PATH = Path(os.environ.get('JD_PRED', OUT_DIR / 'step6_test_predictions.parquet'))     # Step 6
SEASONAL = ['cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share']
LSA = ['u_sem_lsa_7d', 'u_sem_lsa_all', 'slate_pct_sem']                                   # Step 4 / 4b profile columns
CPS_TOP5 = ['cps_max', 'slate_pct_cps_max', 'slate_pct_cps_query', 'cps_top3_new', 'cps_query']   # Step 9's suggested subset
nU = len(pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user']))
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.02 if SMOKE else np.ones(nU, bool)   # same 2% as Step 9 smoke
assert CPS_PATH.exists(), f'{CPS_PATH} not found: run Step 9 first'
print(f'{in_scope.sum():,} records in scope' + (' | SMOKE TEST' if SMOKE else ''))

Mounted at /content/drive
173,831 records in scope


## 1. Load the feature table and the Step 9 columns

The Step 9 table is keyed by `user, wid` and written in `candidates.parquet` order, which is also the order of `features.parquet`; the join is checked to be one-to-one and complete. Slates are shuffled once with the Step 6 seed and slates without any interaction are dropped, as in every earlier benchmark.

In [2]:
BASE = [c for c in pq.read_schema(OUT_DIR / 'features.parquet').names
        if c not in ('user', 'wid', 'label', 'split') and c not in SEASONAL]
flt = ds.field('user').isin(pa.array(np.flatnonzero(in_scope).astype(np.int32))) if SMOKE else None
feat = ds.dataset(OUT_DIR / 'features.parquet').to_table(columns=['user', 'wid', 'label', 'split'] + BASE,
                                                         filter=flt).to_pandas()
cps = pd.read_parquet(CPS_PATH)
CPS = [c for c in cps.columns if c not in ('user', 'wid')]
assert not set(CPS) & set(BASE), 'Step 9 column names collide with features.parquet'
assert set(CPS_TOP5) <= set(CPS), 'Step 9 table lacks the expected columns'
if not set(LSA) <= set(BASE):
    print(f'WARNING: {sorted(set(LSA) - set(BASE))} not in features.parquet (run Steps 4 and 4b first); continuing without them')
    LSA = [c for c in LSA if c in BASE]
n0 = len(feat)
feat = feat.merge(cps, on=['user', 'wid'], how='left', validate='one_to_one')
assert len(feat) == n0 and feat['cps_max'].notna().all(), 'Step 9 table does not cover features.parquet'
del cps; gc.collect()

feat = feat.iloc[np.lexsort((np.random.default_rng(SEED).random(len(feat)), feat['user'].to_numpy()))]
feat = feat[feat.groupby('user')['label'].transform('max') > 0].reset_index(drop=True)
part = {s: feat.index[feat['split'] == s].to_numpy() for s in ['train', 'val', 'test']}
groups = {s: feat.loc[part[s]].groupby('user', sort=False).size().to_numpy() for s in part}
U = {s: feat['user'].to_numpy()[part[s]] for s in part}
Y = {s: feat['label'].to_numpy()[part[s]] for s in part}
hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
print(f'baseline features {len(BASE)} | Step 9 columns {len(CPS)} | candidate rows {len(feat):,} | ' +
      ', '.join(f'{s} {len(g):,} slates' for s, g in groups.items()))

baseline features 55 | Step 9 columns 13 | candidate rows 15,348,717 | train 120,081 slates, val 25,825 slates, test 25,822 slates


## 2. LightGBM variants, chosen on validation

Configuration of the best Step 6 LightGBM (LambdaRank, `label_gain` [0, 1, 3, 7], learning rate 0.03, up to 1,200 trees, 63 leaves, early stopping on validation NDCG@10), identical rows for every variant; only the feature list changes.

| Variant | Features |
|---|---|
| baseline | the 55 Step 6 columns |
| + all 13 | baseline + every Step 9 column |
| + all 13, − LSA | as above without `u_sem_lsa_7d`, `u_sem_lsa_all`, `slate_pct_sem` (Step 9 found they cost nothing once the TF-IDF profiles are in) |
| + top 5, − LSA | baseline − LSA + `cps_max`, `slate_pct_cps_max`, `slate_pct_cps_query`, `cps_top3_new`, `cps_query` (the columns carrying 12 of the 13 columns' split gain in Step 9) |

In [3]:
TREES = 60 if SMOKE else 1200
LGB = dict(objective='lambdarank', n_estimators=TREES, learning_rate=0.03, num_leaves=63, min_child_samples=200,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1,
           label_gain=[0, 1, 3, 7])

def slate_metrics(u, lab, score, k=10, seed=0):
    lab = lab.astype(np.float64)
    order = np.lexsort((np.random.default_rng(seed).random(len(u)), -np.asarray(score, np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    size = np.diff(np.r_[start, len(u_s)])
    rank = np.arange(len(u_s)) - np.repeat(start, size)
    gid = np.repeat(np.arange(len(start)), size)
    disc = np.where(rank < k, 1.0 / np.log2(rank + 2), 0.0)
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    idcg = np.bincount(gid, weights=(2 ** lab[np.lexsort((-lab, u))] - 1) * disc)
    has_buy = np.bincount(gid, weights=lab_s == 3) > 0
    return {'user': u_s[start], 'ndcg': dcg / idcg, 'hit': np.bincount(gid, weights=(lab_s > 0) & (rank < k)) > 0,
            'has_buy': has_buy, 'buy_hit': np.bincount(gid, weights=(lab_s == 3) & (rank < k)) > 0}

def summary(m):
    return {'NDCG@10': m['ndcg'].mean(), 'HR@10': m['hit'].mean(), 'Purchase-HR@10': m['buy_hit'][m['has_buy']].mean()}

BASE_NAME = 'baseline (Step 6 LightGBM)'
NO_LSA = [c for c in BASE if c not in LSA]
VARIANTS = {BASE_NAME: BASE,
            '+ all 13': BASE + CPS,
            '+ all 13, - LSA': NO_LSA + CPS,
            '+ top 5, - LSA': NO_LSA + CPS_TOP5}
grid, SCORES, IMP = {}, {}, {}
for name, cols in VARIANTS.items():
    t0 = time.time()
    X = {s: feat.loc[part[s], cols] for s in part}
    m = lgb.LGBMRanker(**LGB).fit(X['train'], Y['train'], group=groups['train'],
                                  eval_set=[(X['val'], Y['val'])], eval_group=[groups['val']], eval_at=[10],
                                  callbacks=[lgb.early_stopping(50, verbose=False)])
    SCORES[name] = {s: m.predict(X[s]) for s in ('val', 'test')}
    imp = pd.Series(m.booster_.feature_importance('gain'), index=cols); IMP[name] = imp / imp.sum()
    val = summary(slate_metrics(U['val'], Y['val'], SCORES[name]['val']))
    grid[name] = {**{f'val {k}': v for k, v in val.items()}, 'features': len(cols), 'trees': m.best_iteration_,
                  'gain share of Step 9 columns': IMP[name].reindex(CPS).fillna(0).sum(), 'min': (time.time() - t0) / 60}
    print(f"{name}: val NDCG@10 {val['NDCG@10']:.4f} | val Purchase-HR@10 {val['Purchase-HR@10']:.4f} | "
          f"{m.best_iteration_} trees | {grid[name]['min']:.1f} min", flush=True)
    del X, m; gc.collect()
GRID = pd.DataFrame(grid).T
GRID.round(4)

baseline (Step 6 LightGBM): val NDCG@10 0.4153 | val Purchase-HR@10 0.6512 | 893 trees | 11.4 min
+ all 13: val NDCG@10 0.4163 | val Purchase-HR@10 0.6542 | 629 trees | 9.3 min
+ all 13, - LSA: val NDCG@10 0.4172 | val Purchase-HR@10 0.6552 | 670 trees | 10.0 min
+ top 5, - LSA: val NDCG@10 0.4160 | val Purchase-HR@10 0.6532 | 515 trees | 7.6 min


,val NDCG@10,val HR@10,val Purchase-HR@10,features,trees,gain share of Step 9 columns,min
baseline (Step 6 LightGBM),0.4153,0.7167,0.6512,55.0,893.0,0.0000,11.3808
+ all 13,0.4163,0.7191,0.6542,68.0,629.0,0.1376,9.3167
"+ all 13, - LSA",0.4172,0.7213,0.6552,65.0,670.0,0.1344,9.9870
"+ top 5, - LSA",0.4160,0.7193,0.6532,57.0,515.0,0.1277,7.5787


In [4]:
CHOSEN = GRID['val NDCG@10'].idxmax()
print(f'chosen on validation: {CHOSEN}')
print('\nsplit-gain share of each Step 9 column (and the LSA columns it competes with):')
pd.concat({n: IMP[n].reindex(CPS + LSA) for n in VARIANTS}, axis=1).round(4)

chosen on validation: + all 13, - LSA

split-gain share of each Step 9 column (and the LSA columns it competes with):


,baseline (Step 6 LightGBM),+ all 13,"+ all 13, - LSA","+ top 5, - LSA"
cps_max,NaN,0.0679,0.0637,0.0682
cps_mean,NaN,0.0011,0.0012,NaN
cps_query,NaN,0.0052,0.0045,0.0053
cps_long,NaN,0.0011,0.0017,NaN
cps_short,NaN,0.0023,0.0025,NaN
cps_top3_new,NaN,0.0047,0.0048,0.0058
cps_cart,NaN,0.0009,0.0012,NaN
cps_buy,NaN,0.0008,0.0008,NaN
cps_max_cart,NaN,0.0015,0.0015,NaN
cps_max_ord,NaN,0.0007,0.0009,NaN


## 3. Test results with paired bootstrap

First and only use of the test set. Each variant is compared with the baseline on the same slates (2,000 resamples of whole slates; positive difference = variant better; `p` is one-sided). The chosen variant is the result; the other rows are shown for completeness and were not used to choose.

In [5]:
M = {n: slate_metrics(U['test'], Y['test'], SCORES[n]['test']) for n in VARIANTS}

def paired(ma, mb, metric, n_boot=2000, seed=SEED):
    if metric == 'NDCG@10':
        d = ma['ndcg'] - mb['ndcg']
    else:
        d = ma['buy_hit'][ma['has_buy']].astype(float) - mb['buy_hit'][mb['has_buy']].astype(float)
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0, len(d), (200, len(d)))].mean(1) for _ in range(n_boot // 200)])
    return {'difference': d.mean(), 'ci_low': np.percentile(boot, 2.5), 'ci_high': np.percentile(boot, 97.5),
            'p (one-sided)': max((boot <= 0).mean(), 1 / n_boot)}

TEST = pd.DataFrame({n: summary(M[n]) for n in VARIANTS}).T
TESTS = {f'{n} vs baseline [{k}]': paired(M[n], M[BASE_NAME], k)
         for n in VARIANTS if n != BASE_NAME for k in ('NDCG@10', 'Purchase-HR@10')}
print(TEST.round(4).to_string()); print()
pd.DataFrame(TESTS).T.round(5)

                            NDCG@10   HR@10  Purchase-HR@10
baseline (Step 6 LightGBM)   0.4158  0.7192          0.6662
+ all 13                     0.4178  0.7233          0.6724
+ all 13, - LSA              0.4183  0.7227          0.6708
+ top 5, - LSA               0.4165  0.7224          0.6662



,difference,ci_low,ci_high,p (one-sided)
+ all 13 vs baseline [NDCG@10],0.00197,0.00064,0.00326,0.0025
+ all 13 vs baseline [Purchase-HR@10],0.00616,-0.00162,0.01393,0.0575
"+ all 13, - LSA vs baseline [NDCG@10]",0.00243,0.00103,0.00375,0.0005
"+ all 13, - LSA vs baseline [Purchase-HR@10]",0.00454,-0.00325,0.01232,0.1380
"+ top 5, - LSA vs baseline [NDCG@10]",0.00071,-0.00074,0.00206,0.1685
"+ top 5, - LSA vs baseline [Purchase-HR@10]",0.00000,-0.00778,0.00778,0.5180


### Cold, mixed and warm slates

Same segmentation as Steps 3–7: a slate is *all cold* when none of its interacted items appears in any user's history, *all warm* when all do. Step 9's gain came from all-cold slates, which are also where Step 7 located most of the final model's misses.

In [6]:
te = feat.loc[part['test']]
pos = te[te['label'] > 0]
warm = pd.Series(np.isin(pos['wid'].to_numpy(), hist_wids)).groupby(pos['user'].to_numpy()).mean()
warm = warm.reindex(M[BASE_NAME]['user']).to_numpy()
SEGMENT = np.where(warm == 0, 'all cold', np.where(warm == 1, 'all warm', 'mixed'))
rows = []
for seg in ['all cold', 'mixed', 'all warm']:
    mask = SEGMENT == seg
    row = {'segment': seg, 'slates': int(mask.sum())}
    for n in VARIANTS:
        row[n] = M[n]['ndcg'][mask].mean()
    row['chosen - baseline'] = row[CHOSEN] - row[BASE_NAME]
    rows.append(row)
SEG = pd.DataFrame(rows).set_index('segment')
SEG.round(4)

,slates,baseline (Step 6 LightGBM),+ all 13,"+ all 13, - LSA","+ top 5, - LSA",chosen - baseline
segment,,,,,,
all cold,3977,0.1124,0.1232,0.1223,0.1214,0.0100
mixed,3379,0.2702,0.2722,0.2725,0.2719,0.0023
all warm,18466,0.5078,0.5079,0.5087,0.5066,0.0008


## 4. Refresh the final ensemble

Step 6's final model is a within-slate z-score blend, weight 0.5 on the neural ranker, chosen on validation there. The neural test scores saved by Step 6 are reused unchanged and only the LightGBM half is swapped for the chosen variant; the weight is kept, so nothing is chosen on test. The baseline row reproduces the published ensemble as a consistency check (the retrained LightGBM differs from Step 6's only by non-determinism).

In [7]:
def zscore(user, x):
    g = pd.Series(x).groupby(user)
    return (x - g.transform('mean').to_numpy()) / (g.transform('std', ddof=0).to_numpy() + 1e-9)

ENS = None
if PRED_PATH.exists():
    old = pd.read_parquet(PRED_PATH, columns=['user', 'wid', 'neural', 'ensemble'])
    te_keys = pd.DataFrame({'user': U['test'], 'wid': feat['wid'].to_numpy()[part['test']]})
    j = te_keys.merge(old, on=['user', 'wid'], how='left', validate='one_to_one')
    assert j['neural'].notna().all(), 'Step 6 predictions do not cover this test set'
    neural = j['neural'].to_numpy()
    ENS = {'Step 6 ensemble (published)': j['ensemble'].to_numpy(),
           'neural (Step 6)': neural,
           'ensemble: neural + baseline (reproduction)': 0.5 * zscore(U['test'], neural) + 0.5 * zscore(U['test'], SCORES[BASE_NAME]['test']),
           f'new ensemble: neural + {CHOSEN}': 0.5 * zscore(U['test'], neural) + 0.5 * zscore(U['test'], SCORES[CHOSEN]['test'])}
    for n, s in ENS.items():
        M[n] = slate_metrics(U['test'], Y['test'], s)
    ens_names = list(ENS)
    NEW_ENS = ens_names[3]
    ENS_TABLE = pd.DataFrame({n: summary(M[n]) for n in ens_names}).T
    ENS_TABLE['cold-slate NDCG@10'] = [M[n]['ndcg'][SEGMENT == 'all cold'].mean() for n in ens_names]
    for k in ('NDCG@10', 'Purchase-HR@10'):
        TESTS[f'new ensemble vs Step 6 ensemble [{k}]'] = paired(M[NEW_ENS], M[ens_names[0]], k)
        TESTS[f'new ensemble vs neural [{k}]'] = paired(M[NEW_ENS], M[ens_names[1]], k)
        TESTS[f'new ensemble vs {CHOSEN} [{k}]'] = paired(M[NEW_ENS], M[CHOSEN], k)
    print(ENS_TABLE.round(4).to_string()); print()
    display(pd.DataFrame(TESTS).T.iloc[-6:].round(5))
else:
    print(f'{PRED_PATH} not found: run Step 6 first to compare against the final ensemble')

                                            NDCG@10   HR@10  Purchase-HR@10  cold-slate NDCG@10
Step 6 ensemble (published)                  0.4201  0.7236          0.6756              0.1198
neural (Step 6)                              0.4185  0.7235          0.6750              0.1240
ensemble: neural + baseline (reproduction)   0.4201  0.7236          0.6756              0.1198
new ensemble: neural + + all 13, - LSA       0.4216  0.7254          0.6717              0.1244



,difference,ci_low,ci_high,p (one-sided)
new ensemble vs Step 6 ensemble [NDCG@10],0.00148,0.00075,0.00229,0.0005
new ensemble vs neural [NDCG@10],0.00304,0.00204,0.00401,0.0005
"new ensemble vs + all 13, - LSA [NDCG@10]",0.00330,0.00192,0.00472,0.0005
new ensemble vs Step 6 ensemble [Purchase-HR@10],-0.00389,-0.00875,0.00097,0.9525
new ensemble vs neural [Purchase-HR@10],-0.00324,-0.00876,0.00292,0.8470
"new ensemble vs + all 13, - LSA [Purchase-HR@10]",0.00097,-0.00583,0.00778,0.4035


## 5. Save

`step11_chosen_features.json` records the LightGBM feature list chosen on validation; `step11_test_predictions.parquet` holds the new LightGBM and ensemble scores for every test candidate, in the same key as `step6_test_predictions.parquet`.

In [8]:
def safe_write(path, write):
    tmp = path.with_name('tmp_' + path.name)
    write(tmp); os.replace(tmp, path)
    print('saved', path)

res = pd.concat([GRID.assign(stage='LightGBM variants (val)'), TEST.assign(stage='LightGBM variants (test)'),
                 pd.DataFrame(TESTS).T.assign(stage='paired bootstrap (test)'),
                 SEG.assign(stage='segments (test NDCG@10)')]
                + ([ENS_TABLE.assign(stage='ensemble (test)')] if ENS is not None else []))
res.index.name = 'model / comparison'
pred = pd.DataFrame({'user': U['test'], 'wid': feat['wid'].to_numpy()[part['test']], 'label': Y['test'],
                     'lgbm_new': SCORES[CHOSEN]['test']})
if ENS is not None:
    pred['ensemble_new'] = ENS[NEW_ENS]
safe_write(OUT_DIR / f'step11_results{SFX}.csv', lambda p: res.to_csv(p))
safe_write(OUT_DIR / f'step11_test_predictions{SFX}.parquet', lambda p: pred.to_parquet(p, index=False))
safe_write(OUT_DIR / f'step11_chosen_features{SFX}.json',
           lambda p: p.write_text(json.dumps({'chosen': CHOSEN, 'lightgbm_features': VARIANTS[CHOSEN],
                                              'val_ndcg10': GRID.loc[CHOSEN, 'val NDCG@10']}, indent=1)))
try:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted - safe to close the runtime')
except NameError:
    pass

saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step11_results.csv
saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step11_test_predictions.parquet
saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step11_chosen_features.json
Drive flushed and unmounted - safe to close the runtime


## 6. Takeaways

*(to fill after the Colab run)*

- Validation: which variant was chosen, and the gain over the Step 6 LightGBM.
- Test: paired-bootstrap difference for the chosen variant, overall and on all-cold slates.
- Ensemble: new ensemble vs the published 0.4201, and whether the gain survives the blend with the neural ranker.
- Decision: adopt (update Step 6 with `JD_CPS=1`, Section 7) or keep the Step 6 ensemble.

## 7. Carrying the features into the whole ensemble

The ensemble gains from the neural ranker on cold slates, which is also where the Step 9 columns act, so part of the LightGBM-side gain is expected to be absorbed by the blend. The only way to let *both* halves use the columns is to retrain Step 6 with them. `step6_tuning_ensemble.ipynb` now accepts `JD_CPS=1` (set `%env JD_CPS=1` in the first Colab cell): it joins the Step 9 table, gives LightGBM the baseline minus the LSA profile columns plus all 13 `cps_*` columns, gives the neural ranker its 32 lean features plus the five strongest `cps_*` columns, re-chooses the blend on validation, and writes `step6_results_cps.csv` and `step6_test_predictions_cps.parquet` beside the originals. Run it (L4 GPU, about the Step 6 run time) only if this notebook shows a gain on the LightGBM side that is worth the GPU time; if the new ensemble here already matches the neural ranker's cold-slate NDCG@10, the retrain is unlikely to add more.